# **Multi-Asset Portfolio Performance, Risk & Attribution**

**FINM3422 — Assessment 2 - Group 8**

# **1.0 Introduction**

This report presents a performance, risk, and attribution analysis of a large Australian superannuation fund investing across five asset-class sleeves: Australian Equities (AUS EQ), International Equities (INTL EQ), Bonds / Fixed Income, Real Estate (RE), and Private Equity / Venture Capital (PE/VC). Each sleeve is managed by an external manager and evaluated against a designated benchmark index.

The analysis covers the 10-year period from January 2016 to December 2025 using monthly return data, assessed against a composite benchmark constructed from Strategic Asset Allocation (SAA) weights. Performance is evaluated at both sleeve and total fund level, with APRA-inspired supervisory checks applied to assess long-run return adequacy, volatility, drawdown resilience, and stress scenario behaviour. Active return is further decomposed using the Brinson attribution framework to identify the  contribution of allocation and selection decisions across all five sleeves.

# **2.0 Data Overview**

The dataset consists of monthly return series for five asset-class sleeves, covering both manager returns and corresponding benchmark indices over the period January 2016 to December 2025 (120 observations). Benchmarks reflect industry-standard proxies: the S&P/ASX 200 for AUS EQ, MSCI World ex-Australia for INTL EQ, a Bloomberg AusBond proxy for Bonds, an A-REIT proxy for Real Estate, and a stylised synthetic benchmark for PE/VC. A risk-free rate proxy is included for Sharpe ratio calculations.

All data is loaded via a centralised pipeline that enforces a DatetimeIndex, validates monthly frequency, confirms full manager-benchmark date alignment, and checks for missing values prior to analysis.

Two sets of weights are used throughout — the Tactical Asset Allocation (TAA), reflecting the fund's current active positioning, and the Strategic Asset Allocation (SAA), representing the long-term policy benchmark. Both are held constant across the analysis period:

| Asset Class            | TAA Weight | SAA Weight |
|------------------------|------------|------------|
| Australian Equities    | 35%        | 40%        |
| International Equities | 35%        | 30%        |
| Bonds                  | 15%        | 20%        |
| Real Estate            | 5%         | 5%         |
| PE/VC                  | 10%        | 5%         |

### 2.1 Environment Info, Imports and Confit

In [ ]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

sys.path.append("../src")

from data_loader import load_all_data, SLEEVE_LABELS, TAA_WEIGHTS, SAA_WEIGHTS
from performance import (all_sleeves_summary, sleeve_summary, annualised_return, annualised_volatility, max_drawdown, apra_checks, equity_crash, bond_yield_spike, display_summary_tables, fund_vs_benchmark)
from attribution import brinson_attribution
from charts import (plot_sleeve_wealth_index, plot_fund_vs_benchmark, plot_sharpe_bar, plot_ir_bar, plot_apra_drawdown_threshold, plot_attribution)

print(sys.version)
print("pandas:", pd.__version__, "numpy:", np.__version__)
pd.set_option("display.float_format", "{:.4f}".format)

### 2.2 Parameters and Paths

In [ ]:
# Base path to data folder
BASE_path = "/Users/ameliaperry/Desktop/BAFE/FINM3422/python case/FINM3422-Group-8-Task-2-1/data/"

# TAA and SAA weights loaded from data_loader
taa_weights = pd.Series(TAA_WEIGHTS)
saa_weights = pd.Series(SAA_WEIGHTS)

### 2.3 Data Load and Sanity Check

In [ ]:
# Load and Validate Data
data = load_all_data(BASE_path)

df_managers   = data["managers"]
df_benchmarks = data["benchmarks"]
df_rf         = data["rf"]
# Sanity Checks
print("Any nulls in benchmarks?", df_benchmarks.isna().sum().sum())
print("Any nulls in managers?", df_managers.isna().sum().sum())
print("Dates aligned?", df_benchmarks.index.equals(df_managers.index))
print("Monotonic dates?", df_benchmarks.index.is_monotonic_increasing)
#
display(df_managers.head())
display(df_benchmarks.head())
display(df_rf.head()) 